In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb


val_scores = pd.read_parquet('../data/processed/val_scores.parquet')

In [ ]:
s = val_scores.sort_values(by='uplift_t', ascending=False).reset_index(drop=True)
s['is_ad'] = s['treatment'] == 1
s['is_ad_visit'] = (s['treatment'] == 1) & (s['visit'] == 1)
s['is_noad'] = s['treatment'] == 0
s['is_noad_visit'] = (s['treatment'] == 0) & (s['visit'] == 1) 
    
    
s['n_ad'] = s['is_ad'].cumsum()
s['n_ad_visit'] = s['is_ad_visit'].cumsum()
s['n_noad'] = s['is_noad'].cumsum()
s['n_noad_visit'] = s['is_noad_visit'].cumsum()

s['qini'] = (s['n_ad_visit'] - s['n_noad_visit'] * (s['n_ad'] / s['n_noad'])).fillna(0)

#share of the population targeted by the model
s['share'] = (s.index + 1) / len(s)

print('NaN left:', s['qini'].isna().sum())
print('Rows add up:', (s['n_ad'] + s['n_noad'] == s.index + 1).all())
print('Last qini value:', s['qini'].iloc[-1])

ad = s['treatment'] == 1
expected = (s.loc[ad, 'visit'].mean() - s.loc[~ad, 'visit'].mean()) * ad.sum()
print('Expected (ATE x ad users):', expected)


In [ ]:
# Plot every 1,000th row, plus the last one, so it draws quickly.
plot_rows = s.iloc[list(range(0, len(s), 1000)) + [len(s) - 1]]
final = s['qini'].iloc[-1]

plt.figure(figsize=(8, 5))
plt.plot(plot_rows['share'], plot_rows['qini'], label='T-learner')
plt.plot([0, 1], [0, final], linestyle='--', color='gray', label='Random targeting')
plt.xlabel('Share of users targeted')
plt.ylabel('Extra visits caused by the ad')
plt.title('Qini curve on validation (visit)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()